# InstructGPT from first principles, at small scale

**Paper:** Ouyang et al., 2022, *Training language models to follow instructions with human feedback* ([arXiv:2203.02155](https://arxiv.org/abs/2203.02155))

This notebook rebuilds the InstructGPT recipe end to end on a model small enough for a laptop:

| Paper (2022) | This notebook |
|---|---|
| GPT-3 (1.3B / 6B / 175B), a pure next-token predictor | **Pythia** (70M → 1.4B), a pure next-token predictor trained only on the Pile |
| **Step 1 – SFT** on ~13k labeler-written demonstrations | SFT on **Dolly-15k** (~15k demonstrations written by Databricks employees, with task categories much like the paper's) |
| **Step 2 – RM** on ~33k labeler comparisons (ranked K=4..9 outputs) | Reward model on **Anthropic HH-RLHF** single-turn pairs (chosen vs rejected) |
| **Step 3 – PPO-ptx** against the RM, with a KL penalty to the SFT model plus pretraining mix-in | A from-scratch **PPO-style clipped policy gradient** with the same KL-penalised reward and optional pretraining (ptx) loss, using an RLOO baseline instead of a value network |
| Human preference win rates, TruthfulQA, RealToxicity, public NLP benchmarks ("alignment tax") | RM-judged win rates, stop/format metrics, ROUGE-L to references, reference-response likelihood, WikiText perplexity (alignment tax) |

Then we open the models up: **attention patterns, residual stream, MLP activations, logit lens, and direct logit attribution**, comparing *base → SFT → RL* on the **same token sequences** to see what instruction tuning changed inside the network.

### How to run
* **Locally (Apple Silicon / CPU):** `pip install -r requirements.txt`, then run all cells. The profile is picked automatically (see the config cell).
* **On `gpu0` (A100-40GB):** see `README.md` for the SSH-tunnelled Jupyter setup. With CUDA present the `gpu0` profile (Pythia-1.4B) is selected.
* Set `IG_PROFILE=smoke` to run everything in a few minutes with a tiny model, which is useful to check the pipeline before a long run.
* Checkpoints are cached under `checkpoints/`, so re-running the notebook skips training and goes straight to analysis.

## 0. Setup

In [ ]:
import os
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")   # let rare unsupported ops fall back to CPU on Apple Silicon
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

import contextlib, gc, json, math, random, re, time
from dataclasses import dataclass, asdict, replace
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from datasets import load_dataset
from transformers import AutoModel, AutoModelForCausalLM, AutoTokenizer, get_cosine_schedule_with_warmup
from transformers.utils import logging as hf_logging

hf_logging.set_verbosity_error()
pd.set_option("display.max_colwidth", 120)
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.3})

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
# bf16 autocast on CUDA; Apple MPS and CPU train in fp32 (simplest and numerically safest at these sizes).
AMP_DTYPE = torch.bfloat16 if DEVICE == "cuda" and torch.cuda.is_bf16_supported() else None
# Frozen models (base, reference, reward model) are kept in bf16 on CUDA to save memory.
FROZEN_DTYPE = torch.bfloat16 if DEVICE == "cuda" else torch.float32

def autocast():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if AMP_DTYPE else contextlib.nullcontext()

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

def set_attn(model, impl):
    """Switch attention kernel at runtime (eager is needed to read out attention probabilities)."""
    if hasattr(model, "set_attn_implementation"): model.set_attn_implementation(impl)
    else: model.config._attn_implementation = impl

def free_memory():
    gc.collect()
    if DEVICE == "cuda": torch.cuda.empty_cache()
    if DEVICE == "mps": torch.mps.empty_cache()

print("torch", torch.__version__, "| device:", DEVICE, "| amp:", AMP_DTYPE)

### Run profiles

All the knobs live in one `Config`. The profiles trade fidelity for wall-clock time. Approximate timings are for a 32 GB M-series Mac (MPS) and an A100-40GB:

| profile | model | params | SFT | RM | RL | wall-clock |
|---|---|---|---|---|---|---|
| `smoke` | pythia-70m | 70M | 40 steps | 40 steps | 4 steps | ~5 min anywhere (pipeline check only) |
| `local` | pythia-160m | 160M | 500 steps (~half epoch) | 400 steps | 60 steps | ~45 min on M-series |
| `local_410m` | pythia-410m | 410M | 500 steps | 400 steps | 60 steps | ~1.5–2 h on M-series |
| `gpu0` | pythia-1.4b | 1.4B | 2 epochs | 1500 steps | 200 steps | ~1–1.5 h on A100-40GB |

In [ ]:
@dataclass
class Config:
    model_name: str
    max_len: int = 384                 # max tokens of prompt+response during training
    # --- Step 1: SFT
    sft_epochs: float = 1.0
    sft_max_steps: int | None = None   # caps the number of optimizer steps (overrides epochs)
    sft_bs: int = 8
    sft_grad_accum: int = 2
    sft_lr: float = 2e-5
    # --- Step 2: reward model
    rm_steps: int = 400
    rm_bs: int = 8                     # comparison pairs per step
    rm_lr: float = 1e-5
    # --- Step 3: RL
    rl_steps: int = 60
    rl_prompts_per_step: int = 8
    rl_k: int = 4                      # samples per prompt (RLOO baseline needs >= 2)
    rl_lr: float = 1e-6
    rl_beta: float = 0.05              # KL coefficient (paper: 0.02)
    rl_clip: float = 0.2
    rl_ppo_epochs: int = 2
    rl_minibatch: int = 16
    rl_gen_tokens: int = 96
    rl_ptx_coef: float = 0.0           # gamma for PPO-ptx; try 1.0 to see the alignment-tax trade-off
    # --- evaluation and probes
    eval_n: int = 64                   # held-out prompts (half Dolly, half HH)
    gen_tokens: int = 128
    gen_bs: int = 16
    probe_every: int = 50              # log interpretability probe metrics every N optimizer steps
    wikitext_tokens: int = 32_768
    grad_checkpointing: bool = False

PROFILES = {
    "smoke": Config("EleutherAI/pythia-70m", max_len=256, sft_max_steps=40, sft_bs=4, rm_steps=40, rm_bs=4,
                    rl_steps=4, rl_prompts_per_step=4, rl_gen_tokens=32, rl_minibatch=8,
                    eval_n=8, gen_tokens=32, probe_every=10, wikitext_tokens=4096),
    "local": Config("EleutherAI/pythia-160m", sft_max_steps=500),
    "local_410m": Config("EleutherAI/pythia-410m", sft_max_steps=500),
    "gpu0": Config("EleutherAI/pythia-1.4b", max_len=512, sft_epochs=2, sft_bs=16, sft_lr=1e-5,
                   rm_steps=1500, rm_bs=16, rl_steps=200, rl_prompts_per_step=16, rl_minibatch=32,
                   rl_gen_tokens=128, eval_n=200, gen_tokens=160, gen_bs=32, probe_every=100,
                   wikitext_tokens=131_072, grad_checkpointing=True),
}
PROFILE = os.environ.get("IG_PROFILE", "gpu0" if DEVICE == "cuda" else "local")
cfg = PROFILES[PROFILE]
if os.environ.get("IG_MODEL"):     # swap the base model while keeping the profile's budget, e.g. IG_MODEL=HuggingFaceTB/SmolLM2-135M
    cfg = replace(cfg, model_name=os.environ["IG_MODEL"])
REUSE_CHECKPOINTS = True     # set False to force retraining

ROOT = Path.cwd()
RUN_DIR = ROOT / "checkpoints" / f"{PROFILE}-{cfg.model_name.split('/')[-1]}"
SFT_DIR, RM_PATH, RL_DIR = RUN_DIR / "sft", RUN_DIR / "rm.pt", RUN_DIR / "rl"
LOG_DIR, TRACE_DIR = RUN_DIR / "logs", RUN_DIR / "traces"
for d in (RUN_DIR, LOG_DIR, TRACE_DIR): d.mkdir(parents=True, exist_ok=True)

seed_everything(0)
print(PROFILE, asdict(cfg))
print("run dir:", RUN_DIR)

## 1. The paper in one page

```
 Step 1: SFT                        Step 2: Reward model                    Step 3: RL (PPO-ptx)
 prompt ─► labeler writes demo      prompt ─► K model samples               prompt ─► policy samples y
 fine-tune GPT-3 on (x, y)          labelers rank them                      RM scores r(x, y)
 (supervised next-token loss)       train r_θ(x, y) on all K·(K-1)/2 pairs   update policy to maximise reward
                                                                            minus KL to SFT, plus pretrain LM loss
```

**Reward model loss** (paper eq. 1). For a prompt $x$ with preferred completion $y_w$ and dispreferred $y_l$:

$$\mathcal{L}(\theta) = -\frac{1}{\binom{K}{2}}\,\mathbb{E}_{(x,y_w,y_l)}\big[\log \sigma\big(r_\theta(x,y_w) - r_\theta(x,y_l)\big)\big]$$

**RL objective** (paper eq. 2):

$$\text{objective}(\phi) = \mathbb{E}_{(x,y)\sim \pi_\phi^{RL}}\Big[r_\theta(x,y) - \beta \log \frac{\pi_\phi^{RL}(y\mid x)}{\pi^{SFT}(y\mid x)}\Big] + \gamma\,\mathbb{E}_{x\sim D_{\text{pretrain}}}\big[\log \pi_\phi^{RL}(x)\big]$$

* The **KL term** ($\beta$) keeps the policy close to the SFT model, so it can't drift into text the RM was never trained to judge (reward hacking).
* The **ptx term** ($\gamma$) mixes in pretraining gradients to reduce the **alignment tax**, the drop on standard NLP benchmarks after RLHF.

**Headline results to keep in mind:**
1. Labelers preferred the **1.3B InstructGPT** over the **175B GPT-3**: alignment mattered more than 100× scale.
2. SFT does most of the work on *format*; RL adds a further, consistent preference gain.
3. Better truthfulness and less toxicity (when asked to be respectful), but no improvement on bias.
4. The alignment tax is real and mostly recovered by PPO-ptx.
5. Models generalise "follow the instruction" to held-out labelers and to code and non-English prompts, which are rare in the fine-tuning data.

**What we simplify, and why:**
* *Humans → existing datasets.* Dolly stands in for labeler demonstrations and HH-RLHF for labeler rankings. HH has pairs (K=2), not K=4–9 rankings.
* *PPO → PPO-style clipping with an RLOO baseline.* The paper's PPO trains a value network (initialised from the RM) to estimate per-token advantages. We sample K completions per prompt and use the mean reward of the *other* K−1 as the baseline (REINFORCE Leave-One-Out; Ahmadian et al., 2024, show it matches PPO for RLHF). It is fewer moving parts with the same objective.
* *Humans as judges → the RM as judge.* We report RM-judged win rates. This is circular for the RL model (it was optimised against that RM), so we also report RM-independent metrics. Section 9 suggests an LLM-as-judge extension.

## 2. Choosing a base model

Requirements: (a) a **pure next-token predictor** with no instruction tuning and ideally no instruction-style data in pretraining, so the before/after contrast is real; (b) small enough to fine-tune fully on a laptop; (c) well-studied by the interpretability community; (d) a **size ladder**, so the same experiment can be repeated at several scales.

| Candidate | Sizes | Pros | Cons |
|---|---|---|---|
| **Pythia** (EleutherAI) ✅ | 70M, 160M, 410M, 1B, 1.4B, 2.8B… | Trained only on the Pile (2022, essentially no chat data); identical data order across sizes; 154 intermediate checkpoints each; native in TransformerLens and nnsight; a huge interpretability literature | Weaker than modern small models; GPT-NeoX quirks (parallel attention+MLP) |
| GPT-2 | 124M–1.5B | The classic, simplest architecture, best supported by every tool | WebText only, 1024 context, quite weak at instruction following even after SFT |
| SmolLM2 base | 135M, 360M, 1.7B | Much stronger for its size; Llama architecture (RoPE, SwiGLU, GQA), like current LLMs | Pretraining mix includes synthetic, textbook-like and some instruction-ish data, which blurs "base vs instruct" |
| Qwen2.5 / Qwen3 base | 0.5B+ | Very strong | Base checkpoints already follow chat formats surprisingly well (heavy synthetic data), which defeats the purpose |

**Choice: Pythia.** It is the cleanest "just predicts the next token" model, and its parallel-residual architecture makes the residual stream decomposition below exact and easy to read. To repeat the study on a modern architecture, set the environment variable `IG_MODEL=HuggingFaceTB/SmolLM2-135M` (or `-360M`) before launching Jupyter. It keeps the profile's training budget and swaps only the model. The whole notebook has been tested end to end with SmolLM2-135M. Expect its *base* model to already half-answer instructions (it still rarely stops and often continues the template), so the base → SFT contrast is smaller than with Pythia. The helper functions below locate layers, attention and MLP modules generically for GPT-NeoX, GPT-2 and Llama-style models.

In [ ]:
tok = AutoTokenizer.from_pretrained(cfg.model_name)
EOS_ID = tok.eos_token_id
PAD_ID = tok.pad_token_id if tok.pad_token_id is not None else EOS_ID
# Pythia was trained on documents separated by <|endoftext|>, so starting every sequence with it is in-distribution
# and gives the model a clean "position 0" (which becomes an attention sink, as we'll see).
BOS_ID = tok.bos_token_id if tok.bos_token_id is not None else EOS_ID

def load_lm(path, dtype=torch.float32):
    m = AutoModelForCausalLM.from_pretrained(path, dtype=dtype).to(DEVICE)
    m.config.pad_token_id = PAD_ID
    return m

def freeze(model):
    model.eval().requires_grad_(False)
    return model.to(FROZEN_DTYPE)

def n_params(m): return sum(p.numel() for p in m.parameters())

base_model = freeze(load_lm(cfg.model_name))
c = base_model.config
print(f"{cfg.model_name}: {n_params(base_model)/1e6:.0f}M params | layers={c.num_hidden_layers} heads={c.num_attention_heads} "
      f"d_model={c.hidden_size} d_mlp={c.intermediate_size} vocab={c.vocab_size}")
print(f"EOS={EOS_ID!r} {tok.eos_token!r}  PAD={PAD_ID}  BOS={BOS_ID}")

## 3. Data

### 3.1 The prompt template
A base model has no notion of "user" and "assistant", so we invent a plain-text protocol. The model only learns what it means through SFT. The response ends with `<|endoftext|>` (EOS): **learning to emit EOS is learning to stop**, one of the first visible effects of instruction tuning.

```
<|endoftext|>### Instruction:
{instruction}

### Input:            ← only when there is context
{context}

### Response:
{response}<|endoftext|>
```

We tokenise **segment by segment** and record which segment every token belongs to (`bos`, `tmpl`, `instr`, `input`, `resp`). Later this lets us ask, for example, "how much do response tokens attend to the instruction?"

In [ ]:
SEGMENTS = ["bos", "tmpl", "instr", "input", "resp"]

def _ids(text): return tok(text, add_special_tokens=False).input_ids

def encode_prompt(instruction, context="", with_segments=False):
    parts = [("bos", None), ("tmpl", "### Instruction:\n"), ("instr", instruction.strip())]
    if context and context.strip():
        parts += [("tmpl", "\n\n### Input:\n"), ("input", context.strip())]
    parts += [("tmpl", "\n\n### Response:\n")]
    ids, segs = [], []
    for seg, text in parts:
        piece = [BOS_ID] if seg == "bos" else _ids(text)
        ids += piece; segs += [seg] * len(piece)
    return (ids, segs) if with_segments else ids

def encode_response(response):
    return _ids(response.strip()) + [EOS_ID]

def format_prompt(instruction, context=""):   # human-readable version, for display only
    return tok.decode(encode_prompt(instruction, context)[1:])

print(format_prompt("Name three primary colors.") + "Red, yellow and blue." + tok.eos_token)

### 3.2 Datasets
* **Dolly-15k** for SFT demonstrations. It has 8 categories (open_qa, general_qa, classification, closed_qa, brainstorming, information_extraction, summarization, creative_writing) that map closely onto the paper's use-case table (generation 45.6%, open QA 12.4%, brainstorming 11.2%, …).
* **HH-RLHF (helpful-base)** for preference pairs. We keep single-turn conversations and convert them to our template. These prompts are also the RL prompt distribution.
* **WikiText-2** for the ptx mix-in (train) and alignment-tax perplexity (test).

In [ ]:
dolly = load_dataset("databricks/databricks-dolly-15k", split="train").shuffle(seed=0)
dolly = [dict(instruction=r["instruction"], context=r["context"], response=r["response"], category=r["category"]) for r in dolly]
dolly_test, dolly_val, dolly_train = dolly[:300], dolly[300:500], dolly[500:]

def parse_hh(row):
    c, r = row["chosen"], row["rejected"]
    if any(s.count("\n\nHuman:") != 1 or s.count("\n\nAssistant:") != 1 for s in (c, r)):
        return None
    pc, ac = c.rsplit("\n\nAssistant:", 1)
    pr, ar = r.rsplit("\n\nAssistant:", 1)
    ac, ar = ac.strip(), ar.strip()
    if pc != pr or not ac or not ar or ac == ar:
        return None
    return dict(instruction=pc.replace("\n\nHuman:", "").strip(), context="", chosen=ac, rejected=ar, category="hh")

hh = load_dataset("Anthropic/hh-rlhf", data_dir="helpful-base")
hh_train = [p for p in map(parse_hh, hh["train"]) if p]
hh_test = [p for p in map(parse_hh, hh["test"]) if p]
random.Random(0).shuffle(hh_train); random.Random(0).shuffle(hh_test)

wiki = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
WIKI_TRAIN_IDS = _ids("".join(wiki["train"][:20000]["text"]))
WIKI_TEST_IDS = _ids("".join(wiki["test"][:]["text"]))[: cfg.wikitext_tokens]

# HH test (~690 single-turn pairs) is split: [0:100] generation eval prompts, [100:350] RM validation, [350:] RM test
RM_VAL, RM_TEST = slice(100, 350), slice(350, None)
assert cfg.eval_n // 2 <= 100

# Held-out evaluation set: half Dolly (with human reference answers), half HH (reference = the chosen answer)
half = cfg.eval_n // 2
eval_items = ([dict(source="dolly", category=d["category"], instruction=d["instruction"], context=d["context"], reference=d["response"]) for d in dolly_test[:half]]
              + [dict(source="hh", category="hh", instruction=h["instruction"], context="", reference=h["chosen"]) for h in hh_test[:half]])

print(f"dolly train/val/test = {len(dolly_train)}/{len(dolly_val)}/{len(dolly_test)} | "
      f"hh single-turn train/test = {len(hh_train)}/{len(hh_test)} | eval prompts = {len(eval_items)}")
print(pd.Series([d["category"] for d in dolly_train]).value_counts(normalize=True).round(3).to_dict())
print("\nHH example:\n", format_prompt(hh_train[0]["instruction"]), "\n  chosen  :", hh_train[0]["chosen"][:200], "\n  rejected:", hh_train[0]["rejected"][:200])

### 3.3 Probe prompts for tracing
A handful of short prompts with **reference answers** that we follow through every stage of training. Comparing models on *teacher-forced* reference sequences matters: every model then sees **exactly the same tokens**, so differences in attention or activations come from the weights, not from different generated text.

In [ ]:
PROBES = [
    dict(name="sky",      instruction="Explain in one sentence why the sky is blue.", context="",
         reference="Sunlight is scattered by air molecules, and blue light is scattered much more than red light, so the sky looks blue."),
    dict(name="fruits",   instruction="List three fruits that are red.", context="",
         reference="Strawberries, cherries and raspberries."),
    dict(name="translate", instruction="Translate to French: Good morning, my friend.", context="",
         reference="Bonjour, mon ami."),
    dict(name="extract",  instruction="What year did the company launch its first product?",
         context="Acme Robotics was founded in 2009 in Austin. After four years of research it launched its first product, a warehouse robot, in 2013.",
         reference="Acme Robotics launched its first product in 2013."),
]

def encode_probe(p):
    pids, psegs = encode_prompt(p["instruction"], p["context"], with_segments=True)
    rids = encode_response(p["reference"])
    return torch.tensor(pids + rids), psegs + ["resp"] * len(rids)

for p in PROBES:
    ids, segs = encode_probe(p)
    print(p["name"], len(ids), "tokens:", " ".join(f"{t!r}" for t in tok.convert_ids_to_tokens(ids[:14])), "...")

## 4. Shared utilities: batching, losses, generation

Two details matter for correctness:
* **Loss masking.** SFT computes next-token loss only on response tokens (labels = −100 on the prompt), so the model learns to *answer*, not to write instructions. We build labels from segment lengths, never by matching token ids: Pythia's pad token can coincide with EOS in other tokenizers, and we must keep the loss on EOS.
* **Position ids from the attention mask.** With left padding (needed for batched generation), we pass `position_ids = cumsum(mask) - 1` so a token's position doesn't depend on how much padding its row has.

In [ ]:
def position_ids_from_mask(mask):
    return (mask.long().cumsum(-1) - 1).clamp(min=0)

def pad_batch(seqs, side="right", pad=PAD_ID):
    L = max(len(s) for s in seqs)
    ids = torch.full((len(seqs), L), pad, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        s = torch.as_tensor(s)
        sl = slice(L - len(s), L) if side == "left" else slice(0, len(s))
        ids[i, sl] = s; mask[i, sl] = 1
    return ids.to(DEVICE), mask.to(DEVICE)

def forward(model, ids, mask, **kw):
    with autocast():
        return model(input_ids=ids, attention_mask=mask, position_ids=position_ids_from_mask(mask), **kw)

def make_sft_example(d, max_len):
    p, r = encode_prompt(d["instruction"], d.get("context", "")), encode_response(d["response"])
    if len(p) + len(r) > max_len:
        return None      # drop rather than truncate, so every training example ends in EOS
    return p, r

def collate_sft(examples):
    ids, mask = pad_batch([p + r for p, r in examples])
    labels = torch.full_like(ids, -100)
    for i, (p, r) in enumerate(examples):
        labels[i, len(p): len(p) + len(r)] = ids[i, len(p): len(p) + len(r)]
    return ids, mask, labels

def lm_loss(model, ids, mask, labels, reduction="mean"):
    logits = forward(model, ids, mask).logits[:, :-1].float()
    tgt = labels[:, 1:]
    loss = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1), ignore_index=-100, reduction="none").view_as(tgt)
    if reduction == "none":
        return loss, (tgt != -100)
    return loss.sum() / (tgt != -100).sum()

@torch.no_grad()
def response_nll(model, data, bs=8):
    """Mean per-token NLL of reference responses: 'how likely does the model find the human demonstration?'"""
    model.eval()
    exs = [e for e in (make_sft_example(d, cfg.max_len) for d in data) if e]
    tot, cnt = 0.0, 0
    for i in range(0, len(exs), bs):
        loss, m = lm_loss(model, *collate_sft(exs[i:i + bs]), reduction="none")
        tot += loss[m].sum().item(); cnt += m.sum().item()
    return tot / cnt

@torch.no_grad()
def wikitext_ppl(model, block=512):
    model.eval(); nll, n = 0.0, 0
    for i in range(0, len(WIKI_TEST_IDS) - 1, block):
        chunk = [BOS_ID] + WIKI_TEST_IDS[i: i + block]
        ids, mask = pad_batch([chunk])
        labels = ids.clone(); labels[:, 0] = -100
        loss, m = lm_loss(model, ids, mask, labels, reduction="none")
        nll += loss[m].sum().item(); n += m.sum().item()
    return math.exp(nll / n)

@torch.no_grad()
def generate(model, prompt_ids, max_new_tokens, temperature=0.7, top_p=0.9, seed=0, bs=None):
    """Batched sampling. Same seed per batch for every model, so comparisons share randomness."""
    model.eval(); bs = bs or cfg.gen_bs; outs = []
    for i in range(0, len(prompt_ids), bs):
        ids, mask = pad_batch(prompt_ids[i: i + bs], side="left")
        torch.manual_seed(seed + i)
        sampling = dict(do_sample=True, temperature=temperature, top_p=top_p, top_k=0) if temperature > 0 else dict(do_sample=False)
        with autocast():
            g = model.generate(input_ids=ids, attention_mask=mask, max_new_tokens=max_new_tokens,
                               pad_token_id=PAD_ID, eos_token_id=EOS_ID, **sampling)
        for row in g[:, ids.shape[1]:].tolist():
            stopped = EOS_ID in row
            row = row[: row.index(EOS_ID)] if stopped else [t for t in row if t != PAD_ID]
            outs.append(dict(text=tok.decode(row), n_tokens=len(row), stopped=stopped))
    return outs

## 5. Before tuning: what does a pure next-token predictor do with an instruction?

The paper's "GPT (prompted)" baseline gave GPT-3 a few-shot prefix to coax it into instruction-following mode. We try both zero-shot (just our template) and few-shot (two worked examples in front).

In [ ]:
FEW_SHOT_EXAMPLES = [("Give two words that rhyme with cat.", "Hat and bat."),
                     ("What is the capital of Japan?", "The capital of Japan is Tokyo.")]
FEW_SHOT_PREFIX = []
for q, a in FEW_SHOT_EXAMPLES:
    FEW_SHOT_PREFIX += encode_prompt(q)[1:] + _ids(a + "\n\n\n")
FEW_SHOT_PREFIX = [BOS_ID] + FEW_SHOT_PREFIX

def build_prompts(items, few_shot=False):
    ps = [encode_prompt(d["instruction"], d.get("context", "")) for d in items]
    return [FEW_SHOT_PREFIX + p[1:] for p in ps] if few_shot else ps

for few in (False, True):
    print(f"==================== base model, {'few' if few else 'zero'}-shot ====================")
    for p, o in zip(PROBES[:3], generate(base_model, build_prompts(PROBES[:3], few), cfg.gen_tokens)):
        print(f"### {p['instruction']}\n→ {o['text'][:400]!r}\n   [{o['n_tokens']} tokens, emitted EOS: {o['stopped']}]\n")

Typical failure modes of the base model, all of which the paper describes for GPT-3:
* **It continues the document instead of answering.** It writes another `### Instruction:`, keeps going, or answers a different question.
* **It never stops.** It rarely emits EOS, because nothing in the Pile signals "the answer is complete here".
* **Repetition loops**, especially in small models.

Few-shot prompting helps with format but costs context and remains brittle. This motivates SFT.

## 6. Interpretability toolkit: tracing a forward pass

HuggingFace's `output_hidden_states` and `output_attentions` give you *part* of the picture, with sharp edges (e.g. in most models the **last** `hidden_states` entry has the final LayerNorm already applied, unlike all the others). We use explicit **PyTorch forward hooks** instead, which work on any `nn.Module` and capture exactly what we ask for:

| what | where we hook | shape | why it's interesting |
|---|---|---|---|
| attention probs | `output_attentions=True` with the *eager* kernel (SDPA/flash never materialise the T×T matrix) | `[L, H, T, T]` | who reads from whom |
| attention output | `layer.attention` output | `[L, T, d]` | what each attention block *writes* to the residual stream |
| MLP output | `layer.mlp` output | `[L, T, d]` | what each MLP block writes |
| MLP neuron activations | `layer.mlp.act` output | `[L, T, d_mlp]` | individual neurons, the units of "features" in MLP interpretability (for gated/SwiGLU MLPs like SmolLM2's this is `act(gate)`, before multiplying by the up-projection) |
| residual stream | `layer` output | `[L+1, T, d]` | the running sum all blocks read from and write to |

Pythia uses a **parallel residual**: `x_{l+1} = x_l + Attn(LN1(x_l)) + MLP(LN2(x_l))` (GPT-2 and Llama apply them one after the other). Either way, every block only *adds* to the residual stream, so the final residual is *exactly* `embedding + Σ attn_out + Σ mlp_out`. We assert this below, and it is what makes **direct logit attribution** (Section 8.5) an exact decomposition.

In [ ]:
def model_parts(model):
    """Locate the pieces we hook, for GPT-NeoX (Pythia), GPT-2 and Llama-style models."""
    base = model.base_model
    layers = next(getattr(base, n) for n in ("layers", "h", "blocks") if hasattr(base, n))
    final_norm = next(getattr(base, n) for n in ("final_layer_norm", "ln_f", "norm") if hasattr(base, n))
    attn = lambda layer: next(getattr(layer, n) for n in ("attention", "attn", "self_attn") if hasattr(layer, n))
    act = lambda layer: next(getattr(layer.mlp, n) for n in ("act", "act_fn") if hasattr(layer.mlp, n))
    return dict(layers=layers, final_norm=final_norm, attn=attn, act=act, unembed=model.get_output_embeddings())

@dataclass
class Trace:
    tokens: list          # string tokens
    ids: torch.Tensor     # [T]
    segs: list            # segment label per token
    attn: torch.Tensor    # [L, H, T, T]  attention probabilities
    resid: torch.Tensor   # [L+1, T, d]   residual stream: resid[0]=embeddings, resid[l+1]=after layer l
    attn_out: torch.Tensor  # [L, T, d]
    mlp_out: torch.Tensor   # [L, T, d]
    mlp_act: torch.Tensor   # [L, T, d_mlp]
    logits: torch.Tensor    # [T, V]

    def save(self, path): torch.save(asdict(self), path)
    @staticmethod
    def load(path): return Trace(**torch.load(path, weights_only=False))

def _first(x): return x[0] if isinstance(x, tuple) else x

@torch.no_grad()
def trace(model, ids, segs=None):
    """One forward pass over a single sequence, recording everything in the table above."""
    model.eval(); P = model_parts(model)
    cache = {k: [] for k in ("attn_out", "mlp_out", "mlp_act", "resid_post")}
    hooks = []
    for layer in P["layers"]:
        hooks += [P["attn"](layer).register_forward_hook(lambda m, i, o: cache["attn_out"].append(_first(o)[0].float().cpu())),
                  layer.mlp.register_forward_hook(lambda m, i, o: cache["mlp_out"].append(_first(o)[0].float().cpu())),
                  P["act"](layer).register_forward_hook(lambda m, i, o: cache["mlp_act"].append(o[0].float().cpu())),
                  layer.register_forward_hook(lambda m, i, o: cache["resid_post"].append(_first(o)[0].float().cpu()))]
    prev_impl = model.config._attn_implementation
    set_attn(model, "eager")       # SDPA/flash kernels never materialise the attention matrix
    try:
        ids = torch.as_tensor(ids)
        x, mask = pad_batch([ids])
        out = forward(model, x, mask, output_attentions=True, output_hidden_states=True)
    finally:
        for h in hooks: h.remove()
        set_attn(model, prev_impl)
    return Trace(tokens=tok.convert_ids_to_tokens(ids.tolist()), ids=ids.cpu(), segs=segs or ["?"] * len(ids),
                 attn=torch.stack([a[0].float().cpu() for a in out.attentions]),
                 resid=torch.stack([out.hidden_states[0][0].float().cpu()] + cache["resid_post"]),
                 attn_out=torch.stack(cache["attn_out"]), mlp_out=torch.stack(cache["mlp_out"]),
                 mlp_act=torch.stack(cache["mlp_act"]), logits=out.logits[0].float().cpu())

def trace_probe(model, probe):
    ids, segs = encode_probe(probe)
    return trace(model, ids, segs)

t = trace_probe(base_model, PROBES[0])
print({k: tuple(v.shape) for k, v in asdict(t).items() if torch.is_tensor(v)})
recon = t.resid[0] + t.attn_out.sum(0) + t.mlp_out.sum(0)
print("max |final resid - (embed + Σattn + Σmlp)| =", (recon - t.resid[-1]).abs().max().item(), "(≈0: the decomposition is exact)")
print("attention rows sum to 1:", torch.allclose(t.attn.sum(-1), torch.ones(()), atol=1e-3))

### 6.1 Cheap scalar probes we can log *during* training
Full traces are tens of MB, which is too much to take at every step. For training-time observability we reduce each probe to a few scalars that tell a story:
* `ref_nll`: NLL of the reference answer (does the model find good answers likely?)
* `p_eos`: probability of EOS right after the reference answer (has it learned to stop?)
* `attn_instr`: average attention mass that **response** tokens put on **instruction/input** tokens, over all layers and heads
* `attn_bos`: the same, for position 0 (the "attention sink"; heads with nothing useful to do park their attention there)

In [ ]:
def segment_attention(tr, query_seg="resp"):
    """[L, H, n_segments]: attention mass from query tokens in `query_seg` onto each key segment."""
    seg_idx = torch.tensor([SEGMENTS.index(s) for s in tr.segs])
    q = (seg_idx == SEGMENTS.index(query_seg)).nonzero().squeeze(-1)
    onehot = F.one_hot(seg_idx, len(SEGMENTS)).float()          # [T, S]
    return (tr.attn[:, :, q, :] @ onehot).mean(2)                 # [L, H, S]

@torch.no_grad()
def probe_metrics(model):
    rows = []
    for p in PROBES:
        tr = trace_probe(model, p)
        logp = torch.log_softmax(tr.logits, -1)
        resp = [i for i, s in enumerate(tr.segs) if s == "resp"]
        ref_nll = -logp[[i - 1 for i in resp], tr.ids[resp]].mean().item()   # logits at i-1 predict token i
        p_eos = logp[resp[-2], EOS_ID].exp().item()                # after the last answer token, before EOS
        sa = segment_attention(tr).mean((0, 1))
        rows.append(dict(ref_nll=ref_nll, p_eos=p_eos, attn_instr=(sa[2] + sa[3]).item(), attn_bos=sa[0].item()))
    return pd.DataFrame(rows).mean().to_dict()

print("base model probe metrics:", {k: round(v, 3) for k, v in probe_metrics(base_model).items()})

### 6.2 Run logging
Each stage appends JSON lines to `checkpoints/<run>/logs/<stage>.jsonl`. That's enough for plots and diffing runs. To use TensorBoard or W&B instead, replace `RunLog.log` with `writer.add_scalar(...)` or `wandb.log(...)`.

In [ ]:
class RunLog:
    def __init__(self, name, fresh=True):
        self.path = LOG_DIR / f"{name}.jsonl"
        if fresh and self.path.exists(): self.path.unlink()
    def log(self, **row):
        with open(self.path, "a") as f: f.write(json.dumps(row) + "\n")
    def df(self):
        return pd.read_json(self.path, lines=True) if self.path.exists() else pd.DataFrame()

def plot_log(df, cols, title, x="step"):
    cols = [c for c in cols if c in df]
    fig, axes = plt.subplots(1, len(cols), figsize=(3.6 * len(cols), 2.8), squeeze=False)
    for ax, col in zip(axes[0], cols):
        s = df[[x, col]].dropna()
        ax.plot(s[x], s[col], marker="." if len(s) < 40 else None); ax.set_title(col, fontsize=10); ax.set_xlabel(x)
    fig.suptitle(title); fig.tight_layout(); plt.show()

## 7. Step 1: Supervised fine-tuning (SFT)

Plain next-token cross-entropy on the response tokens, with AdamW, cosine decay and gradient clipping. Paper notes: they trained 16 epochs and saw validation loss overfit after epoch 1, yet **more epochs still improved RM score and human preference**. Validation loss is not the metric we actually care about. We train for well under that, and you can test the claim yourself by raising `sft_epochs`.

Every `probe_every` steps we log validation loss and the probe metrics from 6.1, so we can watch the model *learn to stop* and *shift attention onto the instruction* as it happens.

In [ ]:
def batches_forever(items, bs, seed=0):
    rng = random.Random(seed)
    while True:
        order = list(range(len(items))); rng.shuffle(order)
        for i in range(0, len(order) - bs + 1, bs):
            yield [items[j] for j in order[i: i + bs]]

def train_sft(model, cfg):
    train = [e for e in (make_sft_example(d, cfg.max_len) for d in dolly_train) if e]
    print(f"SFT examples: {len(train)} (dropped {len(dolly_train) - len(train)} longer than {cfg.max_len} tokens)")
    steps = cfg.sft_max_steps or int(cfg.sft_epochs * len(train) / (cfg.sft_bs * cfg.sft_grad_accum))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg.sft_lr, weight_decay=0.0, betas=(0.9, 0.95))
    sched = get_cosine_schedule_with_warmup(opt, max(1, int(0.05 * steps)), steps)
    if cfg.grad_checkpointing: model.gradient_checkpointing_enable(); model.config.use_cache = False
    log, data = RunLog("sft"), batches_forever(train, cfg.sft_bs)

    def checkpoint(step):
        row = dict(step=step, val_loss=response_nll(model, dolly_val), **probe_metrics(model))
        log.log(**row); model.train(); return row

    print("step 0:", checkpoint(0))
    for step in (pbar := tqdm(range(1, steps + 1), desc="SFT")):
        model.train(); tot = 0.0
        for _ in range(cfg.sft_grad_accum):
            loss = lm_loss(model, *collate_sft(next(data))) / cfg.sft_grad_accum
            loss.backward(); tot += loss.item()
        gn = torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
        log.log(step=step, train_loss=tot, lr=sched.get_last_lr()[0], grad_norm=gn.item())
        pbar.set_postfix(loss=f"{tot:.3f}")
        if step % cfg.probe_every == 0 or step == steps:
            row = checkpoint(step); pbar.write(f"step {step}: " + " ".join(f"{k}={v:.3f}" for k, v in row.items() if k != "step"))
    if cfg.grad_checkpointing: model.gradient_checkpointing_disable(); model.config.use_cache = True
    del opt; free_memory()
    return log

if REUSE_CHECKPOINTS and (SFT_DIR / "config.json").exists():
    print("Loading cached SFT model from", SFT_DIR); sft_log = RunLog("sft", fresh=False)
else:
    seed_everything(0)
    model = load_lm(cfg.model_name)
    t0 = time.time(); sft_log = train_sft(model, cfg); print(f"SFT took {(time.time() - t0) / 60:.1f} min")
    model.save_pretrained(SFT_DIR); tok.save_pretrained(SFT_DIR)
    del model; free_memory()
sft_model = freeze(load_lm(SFT_DIR))

In [ ]:
df = sft_log.df()
plot_log(df, ["train_loss", "val_loss", "lr", "grad_norm"], "SFT optimisation")
plot_log(df, ["ref_nll", "p_eos", "attn_instr", "attn_bos"], "SFT: probe metrics during training (on fixed reference answers)")

**What to look for:**
* `p_eos` should rise from ~0 toward 1 fast: stopping is one of the first things SFT learns, often within the first few dozen steps.
* `attn_instr` should rise as response tokens learn to read the instruction and input. The base model had no reason to treat these spans specially.
* `val_loss` keeps falling, but per the paper it isn't the target; preference is.

In [ ]:
for p, o in zip(PROBES, generate(sft_model, build_prompts(PROBES), cfg.gen_tokens)):
    print(f"### {p['instruction']}\n→ {o['text'][:400]!r}\n   [{o['n_tokens']} tokens, emitted EOS: {o['stopped']}]\n")

## 8. Step 2: Reward model

As in the paper, the RM is **initialised from the SFT model** with the unembedding removed and a scalar head added. It reads the final hidden state at the last token (the EOS) and outputs one number, trained with the pairwise loss from Section 1.

The paper used a 6B RM even for the 175B policy, because 175B RMs were unstable. Ours is the same size as the policy. The paper also trains on all $\binom{K}{2}$ comparisons from one prompt in a single batch element to avoid overfitting; with K=2 that is one pair per prompt.

In [ ]:
class RewardModel(nn.Module):
    def __init__(self, backbone):
        super().__init__()
        self.backbone = backbone
        h = backbone.config.hidden_size
        self.v_head = nn.Linear(h, 1)
        nn.init.normal_(self.v_head.weight, std=1 / math.sqrt(h + 1)); nn.init.zeros_(self.v_head.bias)

    def forward(self, ids, mask):
        with autocast():
            h = self.backbone(input_ids=ids, attention_mask=mask, position_ids=position_ids_from_mask(mask)).last_hidden_state
        last = (mask * torch.arange(mask.size(1), device=mask.device)).argmax(-1)   # last real token (works for left or right padding)
        h_last = h[torch.arange(len(h), device=h.device), last]
        return self.v_head(h_last.to(self.v_head.weight.dtype)).squeeze(-1).float()

def rm_encode(instruction, response, max_len):
    p, r = encode_prompt(instruction), encode_response(response)
    return (p + r)[:max_len] if len(p) < max_len - 8 else None

def rm_pairs(data, max_len):
    out = []
    for d in data:
        c, r = rm_encode(d["instruction"], d["chosen"], max_len), rm_encode(d["instruction"], d["rejected"], max_len)
        if c and r: out.append((c, r))
    return out

def rm_step(rm, pairs):
    ids, mask = pad_batch([c for c, _ in pairs] + [r for _, r in pairs])
    s = rm(ids, mask); sc, sr = s[: len(pairs)], s[len(pairs):]
    return -F.logsigmoid(sc - sr).mean(), (sc > sr).float().mean(), sc, sr

@torch.no_grad()
def rm_evaluate(rm, pairs, bs=16):
    rm.eval(); sc, sr = [], []
    for i in range(0, len(pairs), bs):
        _, _, c, r = rm_step(rm, pairs[i: i + bs]); sc.append(c); sr.append(r)
    sc, sr = torch.cat(sc).cpu(), torch.cat(sr).cpu()
    return dict(acc=(sc > sr).float().mean().item(), margin=(sc - sr).mean().item()), sc, sr

def train_rm(rm, cfg):
    train, val = rm_pairs(hh_train, cfg.max_len), rm_pairs(hh_test[RM_VAL], cfg.max_len)
    opt = torch.optim.AdamW(rm.parameters(), lr=cfg.rm_lr, weight_decay=0.0)
    sched = get_cosine_schedule_with_warmup(opt, max(1, int(0.05 * cfg.rm_steps)), cfg.rm_steps)
    if cfg.grad_checkpointing: rm.backbone.gradient_checkpointing_enable()
    log, data = RunLog("rm"), batches_forever(train, cfg.rm_bs)
    print(f"RM pairs: train={len(train)} val={len(val)}; epochs ≈ {cfg.rm_steps * cfg.rm_bs / len(train):.2f}")
    for step in (pbar := tqdm(range(1, cfg.rm_steps + 1), desc="RM")):
        rm.train()
        loss, acc, _, _ = rm_step(rm, next(data))
        loss.backward(); torch.nn.utils.clip_grad_norm_(rm.parameters(), 1.0)
        opt.step(); sched.step(); opt.zero_grad(set_to_none=True)
        log.log(step=step, train_loss=loss.item(), train_acc=acc.item())
        if step % cfg.probe_every == 0 or step == cfg.rm_steps:
            m, _, _ = rm_evaluate(rm, val); log.log(step=step, val_acc=m["acc"], val_margin=m["margin"])
            pbar.write(f"step {step}: val_acc={m['acc']:.3f} margin={m['margin']:.3f}")
    if cfg.grad_checkpointing: rm.backbone.gradient_checkpointing_disable()
    del opt; free_memory()
    return log

rm = RewardModel(AutoModel.from_pretrained(SFT_DIR, dtype=torch.float32)).to(DEVICE)
if REUSE_CHECKPOINTS and RM_PATH.exists():
    print("Loading cached RM from", RM_PATH); rm.load_state_dict(torch.load(RM_PATH, map_location=DEVICE)); rm_log = RunLog("rm", fresh=False)
else:
    seed_everything(0); t0 = time.time()
    rm_log = train_rm(rm, cfg); print(f"RM took {(time.time() - t0) / 60:.1f} min")
    # Paper: "the reward model is normalised so that labeler demonstrations achieve a mean score of 0" before RL.
    _, sc, _ = rm_evaluate(rm, rm_pairs(hh_train[:400], cfg.max_len))
    with torch.no_grad(): rm.v_head.bias -= sc.mean().to(DEVICE)
    torch.save(rm.state_dict(), RM_PATH)
rm.eval().requires_grad_(False); rm.to(FROZEN_DTYPE)
plot_log(rm_log.df(), ["train_loss", "train_acc", "val_acc", "val_margin"], "Reward model training")

### 8.1 How good, and how biased, is the RM?
The paper's 6B RM reached ~72% agreement with held-out labelers, and labelers agreed with *each other* ~73% of the time, so preference data is noisy and the ceiling is low. Expect roughly 55–65% from our small RMs.

A classic RM failure is **length bias**: longer answers often win, so the RM learns "longer = better", and RL then exploits it. We compare against a trivial baseline that always prefers the longer answer.

In [ ]:
test_pairs = rm_pairs(hh_test[RM_TEST], cfg.max_len)
m, sc, sr = rm_evaluate(rm, test_pairs)
len_c = torch.tensor([len(c) for c, _ in test_pairs]); len_r = torch.tensor([len(r) for _, r in test_pairs])
longer_wins = ((len_c > len_r).float() + 0.5 * (len_c == len_r).float()).mean().item()
scores, lengths = torch.cat([sc, sr]).float(), torch.cat([len_c, len_r]).float()
corr = torch.corrcoef(torch.stack([scores, lengths]))[0, 1].item()
print(f"RM test accuracy: {m['acc']:.3f}   | 'longer answer wins' baseline: {longer_wins:.3f}   | corr(reward, length) = {corr:+.2f}")
plt.figure(figsize=(4.5, 3)); plt.scatter(lengths, scores, s=4, alpha=0.4); plt.xlabel("sequence length (tokens)"); plt.ylabel("reward"); plt.title("Reward vs length"); plt.show()

## 9. Step 3: RL against the reward model (PPO-style, KL-penalised, optional ptx)

One RL step:
1. **Rollout:** sample K completions for each of B prompts from the current policy (temperature 1, no top-k/top-p, so the samples come from exactly the distribution whose log-probs we compute).
2. **Score:** $R = r_\theta(x,y) - \beta \sum_t \big[\log \pi_{old}(y_t) - \log \pi_{SFT}(y_t)\big]$, the paper's eq. 2 per sequence.
3. **Advantage (RLOO):** $A_i = R_i - \frac{1}{K-1}\sum_{j\ne i} R_j$, then whiten over the batch. This replaces PPO's learned value function.
4. **Update:** a few epochs of the clipped surrogate $-\min(\rho_t A, \text{clip}(\rho_t, 1\pm\epsilon) A)$ with per-token ratio $\rho_t = \pi_\phi(y_t)/\pi_{old}(y_t)$, plus $\gamma\cdot$ LM loss on pretraining text (ptx).

**Watch for:** reward rising while KL grows slowly. If KL explodes or the response length runs to the max, the policy is hacking the RM; raise `rl_beta`.

In [ ]:
def token_logprobs(model, ids, mask, prompt_len):
    logits = forward(model, ids, mask).logits[:, prompt_len - 1: -1].float()
    return torch.log_softmax(logits, -1).gather(-1, ids[:, prompt_len:, None]).squeeze(-1)

@torch.no_grad()
def rollout(policy, items, cfg):
    prompts = [p for p in build_prompts(items) for _ in range(cfg.rl_k)]
    ids, mask = pad_batch(prompts, side="left")
    P = ids.shape[1]
    with autocast():
        seqs = policy.generate(input_ids=ids, attention_mask=mask, do_sample=True, temperature=1.0, top_k=0, top_p=1.0,
                               max_new_tokens=cfg.rl_gen_tokens, pad_token_id=PAD_ID, eos_token_id=EOS_ID)
    resp = seqs[:, P:]
    is_eos = (resp == EOS_ID).long()
    resp_mask = ((is_eos.cumsum(1) - is_eos) == 0).long()    # everything up to and including the first EOS
    return seqs, torch.cat([mask, resp_mask], 1), P, resp_mask, is_eos.any(1)

def minibatches(n, size):
    for i in range(0, n, size): yield slice(i, i + size)

def rl_step(policy, ref, rm, opt, items, ptx_data, cfg):
    policy.eval()
    seqs, mask, P, rmask, stopped = rollout(policy, items, cfg)
    N = len(seqs); rm_f = rmask.float()
    with torch.no_grad():
        old_lp = torch.cat([token_logprobs(policy, seqs[s], mask[s], P) for s in minibatches(N, cfg.rl_minibatch)])
        ref_lp = torch.cat([token_logprobs(ref, seqs[s], mask[s], P) for s in minibatches(N, cfg.rl_minibatch)])
        score = torch.cat([rm(seqs[s], mask[s]) for s in minibatches(N, cfg.rl_minibatch)]).float()
        kl = ((old_lp - ref_lp) * rm_f).sum(1)
        R = (score - cfg.rl_beta * kl).view(-1, cfg.rl_k)
        A = (R - (R.sum(1, keepdim=True) - R) / (cfg.rl_k - 1)).flatten()
        A = (A - A.mean()) / (A.std() + 1e-8)
    policy.train(); clipfrac = []
    for _ in range(cfg.rl_ppo_epochs):
        for s in minibatches(N, cfg.rl_minibatch):
            lp = token_logprobs(policy, seqs[s], mask[s], P)
            ratio = (lp - old_lp[s]).exp()
            a = A[s, None]
            pg = -torch.min(ratio * a, ratio.clamp(1 - cfg.rl_clip, 1 + cfg.rl_clip) * a)
            loss = (pg * rm_f[s]).sum() / rm_f.sum()        # token-mean over the whole batch
            loss.backward()
            clipfrac.append((((ratio - 1).abs() > cfg.rl_clip).float() * rm_f[s]).sum().item())
        if cfg.rl_ptx_coef > 0:
            ids_, mask_, labels_ = next(ptx_data)
            (cfg.rl_ptx_coef * lm_loss(policy, ids_, mask_, labels_)).backward()
        torch.nn.utils.clip_grad_norm_(policy.parameters(), 1.0)
        opt.step(); opt.zero_grad(set_to_none=True)
    return dict(reward=score.mean().item(), kl=kl.mean().item(), objective=R.mean().item(),
                resp_len=rm_f.sum(1).mean().item(), eos_rate=stopped.float().mean().item(),
                clipfrac=sum(clipfrac) / (rm_f.sum().item() * cfg.rl_ppo_epochs)), seqs, P

def ptx_batches(bs=4, block=256, seed=0):
    rng = random.Random(seed)
    while True:
        starts = [rng.randrange(0, len(WIKI_TRAIN_IDS) - block) for _ in range(bs)]
        ids, mask = pad_batch([[BOS_ID] + WIKI_TRAIN_IDS[s: s + block] for s in starts])
        labels = ids.clone(); labels[:, 0] = -100
        yield ids, mask, labels

def train_rl(policy, ref, rm, cfg):
    opt = torch.optim.AdamW(policy.parameters(), lr=cfg.rl_lr, weight_decay=0.0)
    log, prompts, ptx = RunLog("rl"), batches_forever(hh_train[2000:], cfg.rl_prompts_per_step, seed=1), ptx_batches()
    if cfg.grad_checkpointing: policy.gradient_checkpointing_enable()
    for step in (pbar := tqdm(range(1, cfg.rl_steps + 1), desc="RL")):
        stats, seqs, P = rl_step(policy, ref, rm, opt, next(prompts), ptx, cfg)
        log.log(step=step, **stats); pbar.set_postfix(reward=f"{stats['reward']:.2f}", kl=f"{stats['kl']:.2f}")
        if step % max(1, cfg.rl_steps // 5) == 0:
            pbar.write(f"step {step}: {stats}\n   sample: {tok.decode(seqs[0, P:], skip_special_tokens=True)[:200]!r}")
    if cfg.grad_checkpointing: policy.gradient_checkpointing_disable()
    del opt; free_memory()
    return log

if REUSE_CHECKPOINTS and (RL_DIR / "config.json").exists():
    print("Loading cached RL policy from", RL_DIR); rl_log = RunLog("rl", fresh=False)
else:
    seed_everything(0)
    policy = load_lm(SFT_DIR)          # the policy starts from SFT; sft_model (frozen) is the KL reference
    t0 = time.time(); rl_log = train_rl(policy, sft_model, rm, cfg); print(f"RL took {(time.time() - t0) / 60:.1f} min")
    policy.save_pretrained(RL_DIR); del policy; free_memory()
rl_model = freeze(load_lm(RL_DIR))
plot_log(rl_log.df(), ["reward", "kl", "objective", "resp_len", "eos_rate", "clipfrac"], "RL training")

## 10. Evaluation: base vs prompted base vs SFT vs RL

The paper's main metric is **human preference win rate against a fixed baseline** (175B SFT). Ours is the RM-judged win rate against our SFT model, plus metrics that don't depend on the RM:

| metric | what it measures | paper analogue |
|---|---|---|
| `win_vs_sft` | P(RM prefers this model's answer over SFT's on the same prompt) | Fig. 1 preference vs SFT 175B |
| `rm_score` | mean normalised reward (0 ≈ a typical human-chosen answer) | RM score |
| `stops` | fraction of answers ending with EOS before the token limit | "follows explicit constraints" / appropriate length |
| `leaks_template` | output contains `### Instruction` or similar, i.e. it continues the *document* instead of answering | "fails to follow the correct instruction" |
| `rep3` | fraction of repeated 3-grams (degeneration) | n/a |
| `rougeL` | overlap with the reference answer | closed-domain tasks |
| `demo_nll` | NLL of the held-out *human demonstrations* (Dolly val) | n/a (a good SFT diagnostic) |
| `wikitext_ppl` | language-modelling perplexity on generic text | **alignment tax** on public NLP datasets |

For RM scoring and ROUGE we cut each output at the first `###` or `Human:` marker, just as a stop sequence would in an API. This keeps the comparison fair to the base model, while `leaks_template` still records that it happened.

In [ ]:
STOP_RE = re.compile(r"\n?#{2,}\s*(Instruction|Input|Response)|\n\s*Human:", re.I)

def clean(text):
    m = STOP_RE.search(text)
    return (text[: m.start()] if m else text).strip()

def rep3(text):
    w = text.split(); grams = list(zip(w, w[1:], w[2:]))
    return 1 - len(set(grams)) / len(grams) if grams else 0.0

def rouge_l(pred, ref):
    a, b = pred.lower().split(), ref.lower().split()
    if not a or not b: return 0.0
    dp = [0] * (len(b) + 1)
    for x in a:
        prev = 0
        for j, y in enumerate(b, 1):
            cur = dp[j]; dp[j] = prev + 1 if x == y else max(dp[j], dp[j - 1]); prev = cur
    lcs = dp[-1]; p, r = lcs / len(a), lcs / len(b)
    return 0.0 if lcs == 0 else 2 * p * r / (p + r)

@torch.no_grad()
def rm_score_texts(items, texts, bs=16):
    seqs = [rm_encode(d["instruction"], t, cfg.max_len) for d, t in zip(items, texts)]
    out = []
    for i in range(0, len(seqs), bs):
        out += rm(*pad_batch(seqs[i: i + bs])).float().tolist()
    return out

def evaluate(name, model, few_shot=False):
    outs = generate(model, build_prompts(eval_items, few_shot), cfg.gen_tokens, seed=1234)
    cleaned = [clean(o["text"]) for o in outs]
    rows = [dict(model=name, idx=i, source=d["source"], category=d["category"], instruction=d["instruction"], output=c,
                 stops=o["stopped"], leaks_template=bool(STOP_RE.search(o["text"])), n_tokens=o["n_tokens"],
                 rep3=rep3(o["text"]), rougeL=rouge_l(c, d["reference"]))
            for i, (d, o, c) in enumerate(zip(eval_items, outs, cleaned))]
    df = pd.DataFrame(rows); df["rm_score"] = rm_score_texts(eval_items, cleaned)
    return df

MODELS = {"base": (base_model, False), "base few-shot": (base_model, True), "SFT": (sft_model, False), "RL": (rl_model, False)}
results = pd.concat([evaluate(n, m, fs) for n, (m, fs) in tqdm(MODELS.items(), desc="eval")], ignore_index=True)

sft_scores = results[results.model == "SFT"].set_index("idx")["rm_score"]
results["win_vs_sft"] = [float(s > sft_scores[i]) + 0.5 * (s == sft_scores[i]) for i, s in zip(results.idx, results.rm_score)]

lm_metrics = {}
for name, (m, fs) in MODELS.items():
    if not fs:
        lm_metrics[name] = dict(demo_nll=response_nll(m, dolly_val), wikitext_ppl=wikitext_ppl(m))
lm_metrics["base few-shot"] = lm_metrics["base"]

summary = results.groupby("model", sort=False)[["win_vs_sft", "rm_score", "stops", "leaks_template", "n_tokens", "rep3", "rougeL"]].mean()
summary = summary.join(pd.DataFrame(lm_metrics).T)
summary.round(3)

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 3))
for ax, col in zip(axes, ["win_vs_sft", "stops", "rougeL", "wikitext_ppl"]):
    summary[col].plot.bar(ax=ax, color=["#999", "#bbb", "#4c72b0", "#dd8452"]); ax.set_title(col); ax.tick_params(axis="x", rotation=20)
axes[0].axhline(0.5, ls="--", c="k", lw=0.8)
fig.tight_layout(); plt.show()

print("RM score by source:")
results.pivot_table(index="model", columns="source", values=["rm_score", "rougeL"], sort=False).round(3)

In [ ]:
def side_by_side(i):
    d = eval_items[i]
    print(f"[{d['source']}/{d['category']}] {d['instruction'][:300]}")
    if d["context"]: print("  context:", d["context"][:200], "...")
    for name in MODELS:
        r = results[(results.model == name) & (results.idx == i)].iloc[0]
        print(f"\n  --- {name} (rm={r.rm_score:+.2f}, stops={r.stops}) ---\n  {r.output[:350]!r}")
    print("\n  --- reference ---\n ", repr(d["reference"][:350]))

side_by_side(0); print("\n" + "=" * 100 + "\n"); side_by_side(len(eval_items) // 2)

**Interpreting the results (compare with the paper):**
* **Base → SFT** should give the biggest jump in `stops`, `leaks_template` and `win_vs_sft`: the model learns the *protocol*. Few-shot prompting closes part of the gap, as in the paper's "GPT (prompted)" curve.
* **SFT → RL** should raise `rm_score`/`win_vs_sft` (by construction, since RL optimised the RM). Check whether RM-independent metrics agree; if `n_tokens` jumps, suspect length hacking.
* **Alignment tax:** `wikitext_ppl` typically rises after SFT/RL. Re-run RL with `rl_ptx_coef=1.0` to see PPO-ptx recover part of it.
* With small models and small `eval_n`, differences of a few points are noise. Increase `eval_n` on gpu0 and look at confidence intervals before drawing conclusions.

## 11. Inside the models: what did instruction tuning change?

All comparisons below run each model on the **same teacher-forced sequence** (probe prompt + reference answer), so any difference comes from the weights.

In [ ]:
traces = {name: [trace_probe(m, p) for p in PROBES] for name, m in [("base", base_model), ("SFT", sft_model), ("RL", rl_model)]}
for name, trs in traces.items():          # persist for offline analysis
    for p, tr in zip(PROBES, trs):
        tr.save(TRACE_DIR / f"{name}_{p['name']}.pt")
print("saved traces to", TRACE_DIR)
L, H = traces["base"][0].attn.shape[:2]

### 11.1 Attention patterns
Pick a probe, a layer and a head. Rows are query positions (the reading token), columns are keys (what it reads). The dashed lines mark where the response starts. Look for:
* **Attention sinks:** a bright first column, where heads park attention on the BOS token.
* **Previous-token heads:** a bright sub-diagonal (common in early layers).
* **Induction / copy heads:** response tokens attending to the instruction token that comes after an earlier copy of the current token (look at the *translate* and *extract* probes).

In [ ]:
def show_attention(tr, layer, head, ax=None, title=""):
    ax = ax or plt.subplots(figsize=(7, 6))[1]
    A = tr.attn[layer, head].numpy(); T = len(tr.tokens)
    ax.imshow(A, cmap="viridis", vmin=0, vmax=min(1.0, A.max()))
    labels = [t.replace("Ġ", "·").replace("Ċ", "⏎") for t in tr.tokens]
    step = max(1, T // 40)
    ax.set_xticks(range(0, T, step)); ax.set_xticklabels(labels[::step], rotation=90, fontsize=6)
    ax.set_yticks(range(0, T, step)); ax.set_yticklabels(labels[::step], fontsize=6)
    r0 = tr.segs.index("resp")
    ax.axhline(r0 - 0.5, c="w", ls="--", lw=0.7); ax.axvline(r0 - 0.5, c="w", ls="--", lw=0.7)
    ax.set_title(title or f"L{layer} H{head}", fontsize=9); ax.grid(False)

def compare_head(probe_idx, layer, head):
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, name in zip(axes, traces):
        show_attention(traces[name][probe_idx], layer, head, ax, f"{name}: {PROBES[probe_idx]['name']} L{layer} H{head}")
    fig.tight_layout(); plt.show()

compare_head(probe_idx=3, layer=L // 2, head=0)

In [ ]:
def head_grid(tr, layer, title=""):
    cols = 8; rows = math.ceil(H / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(2 * cols, 2 * rows), squeeze=False)
    for h, ax in enumerate(axes.flat):
        ax.axis("off")
        if h < H: ax.imshow(tr.attn[layer, h].numpy(), cmap="viridis"); ax.set_title(f"H{h}", fontsize=8)
    fig.suptitle(title or f"layer {layer}: all heads"); fig.tight_layout(); plt.show()

head_grid(traces["SFT"][3], layer=L - 2, title=f"SFT, probe '{PROBES[3]['name']}', layer {L - 2}")

**Interactive alternative:** [`bertviz`](https://github.com/jessevig/bertviz) renders the same tensors as an interactive head view (`pip install bertviz`).

In [ ]:
try:
    from bertviz import head_view
    tr = traces["SFT"][1]
    head_view([a[None] for a in tr.attn], tr.tokens)
except ImportError:
    print("bertviz not installed (optional): pip install bertviz")

### 11.2 Where do response tokens look? Attention mass by segment, per layer
For every response-token query we sum its attention over each key segment, then average over heads and probes. This is the full-resolution version of the `attn_instr` and `attn_bos` scalars we logged during SFT.

In [ ]:
def seg_profile(name):
    return torch.stack([segment_attention(tr) for tr in traces[name]]).mean(0).mean(1)   # [L, S]

fig, axes = plt.subplots(1, 4, figsize=(16, 3), sharey=False)
for ax, (seg_name, cols) in zip(axes, [("BOS sink", [0]), ("template", [1]), ("instruction + input", [2, 3]), ("response so far", [4])]):
    for name in traces:
        ax.plot(seg_profile(name)[:, cols].sum(-1), label=name, marker=".")
    ax.set_title(f"mass on {seg_name}"); ax.set_xlabel("layer")
axes[0].legend(); fig.tight_layout(); plt.show()

### 11.3 Which heads changed most?
Jensen-Shannon divergence between the base and tuned attention distributions, per head, averaged over response-token queries. Fine-tuning tends to change a *small number* of heads a lot rather than everything a little, and those heads are good candidates for "instruction-reading" heads.

In [ ]:
def js_div(p, q, eps=1e-8):
    m = 0.5 * (p + q)
    return 0.5 * ((p * ((p + eps) / (m + eps)).log()).sum(-1) + (q * ((q + eps) / (m + eps)).log()).sum(-1))

def head_change(a, b):
    out = []
    for ta, tb in zip(traces[a], traces[b]):
        q = [i for i, s in enumerate(ta.segs) if s == "resp"]
        out.append(js_div(ta.attn[:, :, q], tb.attn[:, :, q]).mean(-1))
    return torch.stack(out).mean(0)      # [L, H]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, (a, b) in zip(axes, [("base", "SFT"), ("SFT", "RL")]):
    hc = head_change(a, b); im = ax.imshow(hc.T, aspect="auto", cmap="magma"); plt.colorbar(im, ax=ax)
    ax.set_xlabel("layer"); ax.set_ylabel("head"); ax.set_title(f"attention JS divergence: {a} → {b}"); ax.grid(False)
    top = torch.topk(hc.flatten(), 5).indices
    print(f"{a}→{b} most changed heads:", [(int(i // H), int(i % H), round(hc.flatten()[i].item(), 3)) for i in top])
fig.tight_layout(); plt.show()

### 11.4 Logit lens: when does the model "know" the next token?
Apply the final LayerNorm and unembedding to the residual stream after every layer, as if the network stopped there ([nostalgebraist, 2020](https://www.lesswrong.com/posts/AcKRB8wDpdaN6v6ru/interpreting-gpt-the-logit-lens)). We track the probability of the *correct* next token at each layer, at two interesting places:
* the **first response token**, right after `### Response:\n` (the model has to decide to *answer*)
* the **EOS position**, after the last answer token (the model has to decide to *stop*)

In [ ]:
@torch.no_grad()
def logit_lens(model, tr, pos):
    P = model_parts(model); dt = next(model.parameters()).dtype
    x = tr.resid[:, pos].to(DEVICE, dt)          # [L+1, d]
    return torch.log_softmax(P["unembed"](P["final_norm"](x)).float(), -1).cpu()   # [L+1, V]

MODEL_OBJS = {"base": base_model, "SFT": sft_model, "RL": rl_model}
fig, axes = plt.subplots(1, 2, figsize=(12, 3.2))
for ax, which in zip(axes, ["first response token", "EOS"]):
    for name, m in MODEL_OBJS.items():
        curves = []
        for tr in traces[name]:
            r0 = tr.segs.index("resp")
            pos, target = (r0 - 1, tr.ids[r0]) if which == "first response token" else (len(tr.ids) - 2, EOS_ID)
            curves.append(logit_lens(m, tr, pos)[:, target].exp())
        ax.plot(torch.stack(curves).mean(0), marker=".", label=name)
    ax.set_title(f"logit lens: p(correct token) at {which}"); ax.set_xlabel("layer (0 = embeddings)"); ax.legend()
fig.tight_layout(); plt.show()

tr = traces["SFT"][0]; r0 = tr.segs.index("resp")
ll = logit_lens(sft_model, tr, r0 - 1)
print("SFT, top-1 token per layer at the 'Response:' position:", [tok.decode(int(i)) for i in ll.argmax(-1)])

### 11.5 Direct logit attribution: which components push the model to stop?
The final residual is a sum of components: `embed + Σ attn_out[l] + Σ mlp_out[l]`. LayerNorm is linear *once we freeze its scale* (the std of the full residual at that position), so the logit decomposes **exactly**:

$$\text{logit}_y = \sum_{c} \frac{(c - \bar c)}{\sigma}\,\gamma \cdot W_U[y] \;+\; \beta \cdot W_U[y]$$

Raw logits include a large offset shared by every token (very visible in Pythia), and softmax ignores it. So we attribute the **EOS logit minus the mean logit**, which is just as exact: use $W_U[\text{EOS}] - \overline{W_U}$ in place of $W_U[y]$.

We attribute it at the **end of the reference answer**, where the model should stop. Where did SFT install the "stop now" signal: attention (reading that the answer is complete) or MLPs?

In [ ]:
@torch.no_grad()
def direct_logit_attribution(model, tr, pos, token_id):
    P = model_parts(model); norm = P["final_norm"]
    comps = {"embed": tr.resid[0, pos]}
    for l in range(tr.attn_out.shape[0]):
        comps[f"attn{l}"] = tr.attn_out[l, pos]; comps[f"mlp{l}"] = tr.mlp_out[l, pos]
    x = tr.resid[-1, pos]
    centered = isinstance(norm, nn.LayerNorm)
    mu = x.mean() if centered else 0.0
    sigma = torch.sqrt(((x - mu) ** 2).mean() + norm.eps) if centered else torch.sqrt((x ** 2).mean() + norm.variance_epsilon)
    W_U = P["unembed"].weight.float().cpu()
    gamma, w_u = norm.weight.float().cpu(), W_U[token_id] - W_U.mean(0)    # logit(token) - mean logit
    out = {k: ((((c - c.mean()) if centered else c) / sigma) * gamma @ w_u).item() for k, c in comps.items()}
    out["norm_bias"] = (norm.bias.float().cpu() @ w_u).item() if getattr(norm, "bias", None) is not None else 0.0
    return out

fig, ax = plt.subplots(figsize=(14, 3.2))
width = 0.27
for k, (name, m) in enumerate(MODEL_OBJS.items()):
    dla = pd.DataFrame([direct_logit_attribution(m, tr, len(tr.ids) - 2, EOS_ID) for tr in traces[name]]).mean()
    t0_ = traces[name][0]; pos = len(t0_.ids) - 2
    check = sum(direct_logit_attribution(m, t0_, pos, EOS_ID).values())
    actual = (t0_.logits[pos, EOS_ID] - t0_.logits[pos].mean()).item()
    print(f"{name}: Σ components = {check:.3f} vs actual (EOS logit - mean logit) = {actual:.3f}")
    comps = [c for c in dla.index if c.startswith(("attn", "mlp"))]
    ax.bar(np.arange(len(comps)) + (k - 1) * width, dla[comps].values, width, label=name)
ax.set_xticks(np.arange(len(comps))); ax.set_xticklabels(comps, rotation=90, fontsize=7)
ax.set_title("direct contribution to (EOS logit − mean logit) at the end of the answer, mean over probes"); ax.legend(); plt.show()

### 11.6 Representation drift and weight change, layer by layer
* **Residual cosine similarity**: how similar is each layer's residual stream between two models on the same input? Values near 1 mean "unchanged".
* **Relative weight change** $\|W_{tuned} - W_{base}\| / \|W_{base}\|$ per layer and matrix type.

Fine-tuning is famously low-rank and small (this is why LoRA works), so expect tiny relative changes concentrated in particular layers.

In [ ]:
MODEL_PATHS = {"base": cfg.model_name, "SFT": SFT_DIR, "RL": RL_DIR}

def resid_cos(a, b):
    sims = [F.cosine_similarity(ta.resid, tb.resid, dim=-1).mean(-1) for ta, tb in zip(traces[a], traces[b])]
    return torch.stack(sims).mean(0)

def fp32_params(path):
    """Load weights on CPU in fp32: tiny fine-tuning deltas vanish if we diff bf16 copies."""
    return dict(AutoModelForCausalLM.from_pretrained(path, dtype=torch.float32).named_parameters())

def weight_change(path_a, path_b):
    pa, pb = fp32_params(path_a), fp32_params(path_b); rows = []
    for n, wa in pa.items():
        mt = re.search(r"\.(\d+)\.", n)
        if mt is None or wa.dim() < 2: continue
        kind = ".".join(n.split(".")[-3:-1])      # e.g. attention.query_key_value, mlp.dense_4h_to_h
        rows.append(dict(layer=int(mt.group(1)), kind=kind, rel=((pb[n].float() - wa.float()).norm() / wa.float().norm()).item()))
    return pd.DataFrame(rows).pivot_table(index="kind", columns="layer", values="rel")

fig, axes = plt.subplots(1, 3, figsize=(17, 3.4), gridspec_kw={"width_ratios": [1, 1.4, 1.4]})
for a, b in [("base", "SFT"), ("SFT", "RL"), ("base", "RL")]:
    axes[0].plot(resid_cos(a, b), marker=".", label=f"{a} vs {b}")
axes[0].set_title("residual-stream cosine similarity"); axes[0].set_xlabel("layer"); axes[0].legend()
for ax, (a, b) in zip(axes[1:], [("base", "SFT"), ("SFT", "RL")]):
    wc = weight_change(MODEL_PATHS[a], MODEL_PATHS[b])
    im = ax.imshow(wc.values, aspect="auto", cmap="viridis"); plt.colorbar(im, ax=ax)
    ax.set_yticks(range(len(wc.index))); ax.set_yticklabels(wc.index, fontsize=8); ax.set_xlabel("layer")
    ax.set_title(f"relative weight change {a} → {b}"); ax.grid(False)
fig.tight_layout(); plt.show()

### 11.7 MLP neurons that changed behaviour
Find the neurons whose mean activation on response tokens changed most between base and SFT, then look at which tokens they fire on. A neuron that fires on the last few answer tokens in SFT but not in base could be part of the "answer is complete → emit EOS" circuit. *Caveat:* neurons are often polysemantic. Sparse autoencoders (SAEs) are the modern tool for finding cleaner features (see Section 12).

In [ ]:
def response_acts(name):
    return torch.cat([tr.mlp_act[:, [i for i, s in enumerate(tr.segs) if s == "resp"]] for tr in traces[name]], 1)   # [L, Nresp, d_mlp]

base_acts = response_acts("base")
# standardise by each neuron's spread on these tokens, so a few always-large neurons don't dominate the ranking
delta = (response_acts("SFT").mean(1) - base_acts.mean(1)) / (base_acts.std(1) + 1e-3)        # [L, d_mlp]
top = torch.topk(delta.abs().flatten(), 6).indices
d_mlp = delta.shape[1]
for idx in top.tolist():
    l, n = divmod(idx, d_mlp)
    tr_b, tr_s = traces["base"][0], traces["SFT"][0]
    acts = tr_s.mlp_act[l, :, n]
    hot = torch.topk(acts, 5).indices.tolist()
    print(f"L{l:>2} N{n:<5} Δ={delta[l, n]:+.2f}σ | SFT fires most on: " +
          ", ".join(f"{tr_s.tokens[i]!r}({acts[i]:.2f}, base {tr_b.mlp_act[l, i, n]:.2f})" for i in sorted(hot)))

### 11.8 Trace your own prompt
Generate with the SFT model, then trace **all three models on that same token sequence** and compare. Change `MY_PROMPT` and re-run.

In [ ]:
MY_PROMPT = dict(name="mine", instruction="Write a haiku about debugging code.", context="")
gen = generate(sft_model, build_prompts([MY_PROMPT]), cfg.gen_tokens, temperature=0)[0]
MY_PROMPT["reference"] = gen["text"]
print("SFT says:", repr(gen["text"]))
my_traces = {name: trace_probe(m, MY_PROMPT) for name, m in MODEL_OBJS.items()}
for name, tr in my_traces.items():
    sa = segment_attention(tr).mean((0, 1))
    r0 = tr.segs.index("resp")
    nll = -torch.log_softmax(tr.logits[r0 - 1:-1], -1)[torch.arange(len(tr.ids) - r0), tr.ids[r0:]].mean()
    print(f"{name:>4}: NLL of SFT's answer = {nll:.3f} | attn on instr = {(sa[2] + sa[3]):.3f} | attn on BOS = {sa[0]:.3f}")
head_grid(my_traces["SFT"], layer=L // 2, title=f"SFT on your prompt, layer {L // 2}")

## 12. Going further

**Experiments that map directly to paper claims**
1. **Scale vs alignment:** run `smoke`/`local`/`local_410m`/`gpu0` and plot `win_vs_sft` of *base few-shot at the largest size* vs *SFT at the smallest*. Does a small tuned model beat a 10× larger prompted one (the paper's headline)?
2. **SFT epochs:** set `sft_epochs=4` on gpu0. Validation loss overfits; does RM score keep improving, as the paper reports?
3. **Alignment tax and PPO-ptx:** compare `wikitext_ppl` for `rl_ptx_coef` in {0, 1, 4}.
4. **KL coefficient:** sweep `rl_beta` in {0.0, 0.02, 0.1}. With β=0, watch reward hacking (length, repetition, odd tokens that the RM loves).
5. **DPO** (Rafailov et al., 2023) optimises the same KL-regularised objective directly from preference pairs, with no RM and no sampling. It is ~30 lines added to Section 9 and makes a good comparison against RLHF.
6. **LLM-as-judge:** replace the RM win rate with a stronger model judging pairwise (randomise order to control position bias). This approximates the paper's human labelers and removes the circularity of judging RL with its own RM.
7. **Benchmarks:** on gpu0, run [`lm-evaluation-harness`](https://github.com/EleutherAI/lm-evaluation-harness) (`truthfulqa_mc2`, `hellaswag`, `lambada_openai`) on `checkpoints/*/sft` and `rl` for a real alignment-tax table.

**Deeper interpretability tooling**
* **TransformerLens** (`HookedTransformer.from_pretrained("pythia-410m", hf_model=...)`) gives named hook points for every intermediate (`blocks.5.attn.hook_pattern`, `hook_z`, …) plus activation patching utilities. Load our fine-tuned weights through `hf_model=`.
* **nnsight** gives the same with less code and works for remote or large models.
* **Activation patching:** copy SFT activations into the base model one component at a time (e.g. a single head's output at the EOS position) and measure how much of the SFT behaviour (p(EOS)) is restored. This finds components that are *causally* responsible, not just correlated.
* **Pythia checkpoints:** `revision="step3000"` etc. let you ask how the heads SFT relies on (Section 11.3) formed during pretraining.
* **Sparse autoencoders** (SAELens) on the residual stream to find interpretable features that turn on after instruction tuning.
* **Logging at scale:** for many prompts, store compressed summaries (segment attention `[L,H,S]`, logit-lens curves) rather than full `[L,H,T,T]` tensors, and save full traces only for a fixed probe set, as we did here.